# WBCAtt+：现阶段可实现功能（SUPPORTED_ONLY_V2）

请先阅读同包 `GitHub_部署指南.md`。代码用 Git 克隆，完整数据包从同一版本的 GitHub Release 下载到服务器 `/home/jovyan/work/`，无需通过 JupyterLab 上传大文件。本 Notebook 可留在 Windows，用 VS Code 选择已连接的 **Python (wbcattplus server)** 远程内核执行。

逐格按 Shift+Enter，等上一格结束再运行下一格。红色报错立即停止。请先在服务器终端下载 Release 中的完整 ZIP 并核对 SHA-256；Git clone 本身不含数据与离线依赖。本流程不重装 PyTorch/CUDA，不做训练。

**功能边界：** 本包仅部署数据、校验完整性、统计像素尺寸和五类实际前景面积、检查官方集合重复风险并输出报告。患者/切片严格划分、颗粒像素面积及物理单位换算均不实现，仅备注原因。没有第 9 步执行开关。

## 第 1 步：确认远程内核与既有环境（只检查）

In [ ]:
import sys
import platform
from pathlib import Path

print("System:", platform.system(), platform.machine())
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
assert platform.system() == "Linux", "请选择服务器内核，不是 Windows 本地内核"
assert platform.machine().lower() in ("x86_64", "amd64")
assert sys.version_info[:2] == (3, 9), "请选择 wbcattplus，不是 base Python 3.13"
assert Path(sys.prefix).resolve() == Path("/home/jovyan/work/envs/wbcattplus").resolve(), "环境不对，请更换内核"

UPLOAD = Path("/home/jovyan/work/WBCAttPlus_Deployment.zip")
PROJECT = Path("/home/jovyan/work/WBCAttPlus_workspace")

# 本次任务全程 CPU：不导入 torch、不做 GPU 检查、不训练模型。
print("Supported-only audit uses CPU; GPU/PyTorch are not required.")
print("Downloaded release ZIP exists:", UPLOAD.is_file())


## 第 2 步：核验并展开完整 Release 包
先按 GitHub_部署指南完成服务器下载与 SHA-256 校验。以下代码还会复核固定清单及逐文件摘要，不覆盖已有的不同文件。

In [ ]:
"""Standard-library-only, fail-closed extraction for the one-upload package."""

import hashlib
import json
import stat
import zipfile
from pathlib import Path, PurePosixPath


def file_sha256(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def clean_name(name):
    path = PurePosixPath(name)
    if (not name or path.is_absolute() or '..' in path.parts or
            '\\' in name or ':' in name or path.as_posix() != name):
        raise ValueError('Unsafe package path: ' + name)
    return path


def no_symlinks(path):
    for item in (path,) + tuple(path.parents):
        if item.is_symlink():
            raise ValueError('Refusing symlink destination: ' + str(item))


def unpack_package(archive, project, manifest_sha256):
    """Validate all entries before writing; reuse identical files, never clobber."""
    archive, project = Path(archive), Path(project).absolute()
    if not archive.is_file():
        raise FileNotFoundError('Download the full GitHub Release asset first: ' + str(archive))
    no_symlinks(project)
    with zipfile.ZipFile(archive) as bundle:
        infos = bundle.infolist()
        names = [entry.filename for entry in infos]
        if len(names) != len(set(names)):
            raise ValueError('Duplicate ZIP entries')
        for entry in infos:
            clean_name(entry.filename)
            if entry.is_dir() or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError('Unexpected directory/link ZIP entry')
        manifest_bytes = bundle.read('package_manifest.json')
        if hashlib.sha256(manifest_bytes).hexdigest() != manifest_sha256:
            raise ValueError('Package manifest SHA-256 mismatch; do not continue')
        manifest = json.loads(manifest_bytes)
        entries = manifest['files']
        checks = {item['path']: item for item in entries}
        if len(checks) != len(entries):
            raise ValueError('Duplicate manifest paths')
        extras = {'package_manifest.json', 'WBCAttPlus_Setup.ipynb', 'README.md'}
        if set(names) != set(checks) | extras:
            raise ValueError('ZIP members differ from the fixed package manifest')
        # Validate CRC, size, digest and all existing destination conflicts first.
        for entry in infos:
            target = project.joinpath(*clean_name(entry.filename).parts)
            no_symlinks(target)
            expected = checks.get(entry.filename)
            value, size = hashlib.sha256(), 0
            with bundle.open(entry) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''):
                    value.update(block)
                    size += len(block)
            digest = value.hexdigest()
            if expected and (size != expected['size_bytes'] or digest != expected['sha256']):
                raise ValueError('Payload checksum failed: ' + entry.filename)
            if target.exists() and (not target.is_file() or file_sha256(target) != digest):
                raise ValueError('Existing file differs and was preserved: ' + str(target))
        reused = 0
        for entry in infos:
            target = project.joinpath(*clean_name(entry.filename).parts)
            if target.exists():
                reused += 1
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(entry) as source, target.open('xb') as destination:
                for block in iter(lambda: source.read(1024 * 1024), b''):
                    destination.write(block)
    print('Package verified and unpacked:', project, flush=True)
    print('Identical existing files reused:', reused, flush=True)
    return project

MANIFEST_SHA256 = '4dd2d1a84e33460f3949bff369b2bc9790f07c7a0c9d48599f9e07412fa420fe'
unpack_package(UPLOAD, PROJECT, MANIFEST_SHA256)


## 第 3 步：初始化日志助手、复核已展开项目
后续子进程都用当前远程 Python。任何失败都会同时给出日志位置与原始错误末尾。

In [ ]:
import json
import importlib
sys.path.insert(0, str(PROJECT / "scripts"))
import dataset_workflow as workflow
importlib.reload(workflow)

def run_step(step):
    command = [sys.executable, "-u", str(PROJECT / "scripts/dataset_workflow.py"), step,
               "--project", str(PROJECT), "--site", "server"]
    return workflow.run_logged(PROJECT, command, "step_" + step)

run_step("verify")


## 第 4 步：展开并复核 10,298 对图像与 Mask
NFS 小文件较多，可能需要等待。每 4,000 个文件打印进度；相同文件重跑会复用。

In [ ]:
run_step("extract")


## 第 5 步：离线安装体检依赖
只读取包内 wheel，不访问 PyPI，不安装 torch/CUDA。后续统计用新的子进程，不必重启本内核。

In [ ]:
run_step("install")


## 第 6 步：运行合成测试
看到最后的 `OK` 再继续；测试只验证统计、文件检查与功能边界，不生成真实数据的新划分。

In [ ]:
run_step("test")


## 第 7 步：重新逐图体检并生成 PDF
所有统计从服务器文件重新计算。每次运行写入一个新结果目录，不覆盖旧报告。完整性通过不等于官方集合无重复或来源可核验。

In [ ]:
run_step("audit")


## 第 8 步：查看本次结果
只读 latest_run.json 指向的本次输出。PDF/CSV 可在浏览器 JupyterLab 文件面板找到后右键下载，路径在下方打印。

In [ ]:
state = json.loads((PROJECT / "output/latest_run.json").read_text(encoding="utf-8"))
print(json.dumps(state, ensure_ascii=False, indent=2))
assert state["status"] == "PASS", "本次尚未通过，请检查日志，不要读取旧报告当作成功"
assert state["execution_site"] == "server"
RESULT = Path(state["output"])
summary = json.loads((RESULT / "audit_summary.json").read_text(encoding="utf-8"))
receipt = json.loads(Path(state["receipt"]).read_text(encoding="utf-8"))
print("Images / masks / valid pairs:", summary["image_count"], summary["mask_count"], summary["fully_valid_image_mask_pairs"])
print("Resolution:", summary["resolution"])
print("Official split leakage:", summary["official_split_leakage_status"])
print("Strict source grouping:", summary["source_grouping"]["status"])
print("Receipt:", state["receipt"])
print("PDF:", RESULT / "显微数据体检单.pdf")
print("Unsupported feature notes:", RESULT / "功能范围与限制.md")
for item in summary["not_implemented_features"]:
    print(item["feature"], item["status"], item["reason_cn"])
assert summary["package_scope"] == "SUPPORTED_ONLY_V2"
assert receipt["status"] == "FILE_INTEGRITY_PASS"
assert receipt["execution_site"] == "server" and receipt["pdf_created"]
from IPython.display import Image, display
display(Image(filename=str(RESULT / "pixel_area_long_tail.png")))


## 未实现事项备注（不是运行步骤）

- **患者/切片严格划分：未实现。** 无逐图真实来源映射，不能证明无来源泄漏。仅统计官方既有集合，不输出新划分。
- **颗粒像素面积：未实现。** 颗粒只有图像级属性，无独立颗粒 Mask；不会虚构像素面积或以 0 代替缺失。
- **物理分辨率/面积：未实现。** 无显微标定，仅统计宽高 px 和像素个数，不换算 μm/px 或 μm²。

详见同包及每次输出的 `功能范围与限制.md`。这三项未实现不影响支持范围内的体检通过；文件完整性 PASS 不代表患者/切片独立或论文完整复现。